In [4]:
import numpy as np
import pandas as pd
import math

In [50]:
rating_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-ML0321EN-Coursera/labs/v2/module_3/ratings.csv"
rating_df = pd.read_csv(rating_url)
rating_df.head()

,user,item,rating
0,1889878,CC0101EN,5
1,1342067,CL0101EN,3
2,1990814,ML0120ENv3,5
3,380098,BD0211EN,5
4,779563,DS0101EN,3


In [51]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(rating_df, test_size=0.2, random_state=42)

In [52]:
train_sparse_df = train_df.pivot(index='user', columns='item', values='rating').fillna(0).rename_axis(index=None, columns=None)
train_sparse_df.head()

,AI0111EN,BC0101EN,BC0201EN,BC0202EN,BD0101EN,BD0111EN,BD0115EN,BD0121EN,BD0123EN,BD0131EN,...,SW0201EN,TA0105,TA0105EN,TA0106EN,TMP0101EN,TMP0105EN,TMP0106,TMP107,WA0101EN,WA0103EN
2,0.0,4.0,0.0,0.0,5.0,4.0,0.0,5.0,0.0,3.0,...,0.0,5.0,0.0,4.0,0.0,3.0,3.0,0.0,5.0,0.0
4,0.0,0.0,0.0,0.0,0.0,3.0,4.0,5.0,3.0,4.0,...,0.0,4.0,0.0,0.0,0.0,3.0,3.0,0.0,0.0,3.0
5,3.0,0.0,5.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,4.0,4.0,4.0,4.0,4.0,5.0,0.0,3.0
7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [53]:
X = train_sparse_df.values
user_ids = train_sparse_df.index
item_ids = train_sparse_df.columns
train_users = set(user_ids)
train_items = set(item_ids)
test_mask = test_df.user.isin(train_users) & test_df.item.isin(train_items)
test_known_df = test_df[test_mask].copy()
global_mean = train_df['rating'].mean()
n_users = len(user_to_index)
n_items = len(item_to_index)

In [54]:
def train_matrix_factorization(user_to_index, item_to_index, n_factors=20, lr=0.01, reg=0.02, n_epochs=10, random_state=42, shuffle=True):
    rng = np.random.default_rng(random_state)
    U = rng.normal(loc=0.0, scale=0.1, size=(n_users, n_factors))
    I = rng.normal(loc=0.0, scale=0.1, size=(n_items, n_factors))

    for epoch in range(n_epochs):
        if shuffle:
            epoch_df = train_df.sample(frac=1, random_state=random_state + epoch).reset_index(drop=True)
        else:
            epoch_df = train_df.reset_index(drop=True)
        for _, row in epoch_df.iterrows():
            user_id = row['user']
            item_id = row['item']
            rating = row['rating']

            u_idx = user_to_index[user_id]
            i_idx = item_to_index[item_id]

            u_vec = U[u_idx].copy()
            i_vec = I[i_idx].copy()

            pred = np.dot(u_vec, i_vec)
            err = rating - pred

            U[u_idx] = u_vec +  lr * (err * i_vec - reg * u_vec)
            I[i_idx] = i_vec +  lr * (err * u_vec - reg * i_vec)
        return U, I

In [61]:
def predict_mf(user_id, item_id, U, I, user_to_index, item_to_index):
    u_idx = user_to_index[user_id]
    i_idx = item_to_index[item_id]
    return np.clip(np.dot(U[u_idx], I[i_idx]), 1, 5)

In [85]:
user_ids_u = train_df['user'].unique()
item_ids_u = train_df['item'].unique()
user_to_index_u = {user: idx for idx, user in enumerate(user_ids_u)}
item_to_index_u = {item: idx for idx, item in enumerate(item_ids_u)}

U, I = train_matrix_factorization(
    user_to_index=user_to_index_u,
    item_to_index=item_to_index_u,
    n_factors=20,
    lr=0.05,
    reg=0.02,
    n_epochs=100,
    random_state=42,
)

In [86]:
row = test_known_df.iloc[0]
pred = predict_mf(row['user'], row['item'], U, I, user_to_index_u, item_to_index_u)
print(row['rating'], pred)

3 3.192565544943063


In [87]:
y_true = test_known_df['rating'].values
y_pred = []

for _, row in test_known_df.iterrows():
    y_pred.append(predict_mf(row['user'], row['item'], U, I, user_to_index_u, item_to_index_u))

In [88]:
from sklearn.metrics import mean_squared_error

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
print(rmse)

1.0617516111142289
